In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [8]:
stn = pd.read_csv("stations.csv", parse_dates=["commissioned_date","decommissioned_date",
        "firmware_updated_date","competitor_within_1_5km_since"])
stn = stn[~stn.station_id.str.startswith("STN-TST")].copy()   # drop test stations
stn["fixed_cost"] = stn.monthly_rent_inr + stn.monthly_maintenance_inr
stn["has_competitor"] = stn.competitor_within_1_5km_since.notna()
stn["age_months"] = (pd.Timestamp("2025-06-30") - stn.commissioned_date).dt.days/30

print(pd.crosstab(stn.city, stn.expansion_wave))              # where did waves land?
print(pd.crosstab(stn.charger_generation, stn.connectivity_tier))   # confounding check
print(pd.crosstab(stn.location_type, stn.charger_generation))
print(stn.groupby("location_type").agg(n=("station_id","size"), cost=("fixed_cost","mean"),
      slots2w=("slots_2w","mean"), slots3w=("slots_3w","mean")))
print(stn.groupby("city").has_competitor.mean())

expansion_wave  Launch  Wave1_2024H2  Wave2_2025H1
city                                              
Bengaluru           20             6             6
Delhi NCR           18             6             6
Hyderabad           16             5             5
Jaipur              10             4             4
Mumbai              13             4             5
Pune                13             5             4
connectivity_tier   fair  good  poor
charger_generation                  
Gen1                  11    33     7
Gen2                  11    40    11
Gen3                   7    25     5
charger_generation  Gen1  Gen2  Gen3
location_type                       
commercial_hub        23    17     4
highway_fuel_pump      0    14     7
market                18    15    14
residential            4    10     7
tech_park              3     3     1
transit_hub            3     3     4
                    n          cost    slots2w   slots3w
location_type                                         

In [13]:
stn = pd.read_csv(f"stations.csv", parse_dates=["commissioned_date","decommissioned_date",
        "firmware_updated_date","competitor_within_1_5km_since"])
stn = stn[~stn.station_id.str.startswith("STN-TST")].copy()   # drop test stations
stn["fixed_cost"] = stn.monthly_rent_inr + stn.monthly_maintenance_inr
stn["has_competitor"] = stn.competitor_within_1_5km_since.notna()
stn["age_months"] = (pd.Timestamp("2025-06-30") - stn.commissioned_date).dt.days/30

print(pd.crosstab(stn.city, stn.expansion_wave))              # where did waves land?
print(pd.crosstab(stn.charger_generation, stn.connectivity_tier))   # confounding check
print(pd.crosstab(stn.location_type, stn.charger_generation))
print(stn.groupby("location_type").agg(n=("station_id","size"), cost=("fixed_cost","mean"),
      slots2w=("slots_2w","mean"), slots3w=("slots_3w","mean")))
print(stn.groupby("city").has_competitor.mean())


expansion_wave  Launch  Wave1_2024H2  Wave2_2025H1
city                                              
Bengaluru           20             6             6
Delhi NCR           18             6             6
Hyderabad           16             5             5
Jaipur              10             4             4
Mumbai              13             4             5
Pune                13             5             4
connectivity_tier   fair  good  poor
charger_generation                  
Gen1                  11    33     7
Gen2                  11    40    11
Gen3                   7    25     5
charger_generation  Gen1  Gen2  Gen3
location_type                       
commercial_hub        23    17     4
highway_fuel_pump      0    14     7
market                18    15    14
residential            4    10     7
tech_park              3     3     1
transit_hub            3     3     4
                    n          cost    slots2w   slots3w
location_type                                         

In [11]:
import os

os.makedirs("processed", exist_ok=True)

stn.to_csv("processed/stations_clean.csv", index=False)

print("File saved successfully!")
stn.to_csv("processed/stations_clean.csv", index=False)

File saved successfully!


In [14]:
bat = pd.read_csv(f"batteries.csv", parse_dates=["manufacture_date","commission_date","retired_date"])
bat["soh_drop"] = (bat.initial_soh_pct - bat.current_soh_pct).clip(lower=0)
end = bat.retired_date.fillna(pd.Timestamp("2025-06-30"))
bat["age_months"] = ((end - bat.commission_date).dt.days/30).clip(lower=1)
bat["drop_per_month"] = bat.soh_drop / bat.age_months
bat["retired"] = bat.retired_date.notna()

sup = bat.groupby(["supplier","pack_type"]).agg(packs=("battery_id","size"),
        start_soh=("initial_soh_pct","mean"), now_soh=("current_soh_pct","mean"),
        drop_pm=("drop_per_month","mean"), retired=("retired","mean"), cost=("purchase_cost_inr","mean"))
print(sup)
lot = bat.groupby(["supplier","manufacturing_lot"]).agg(packs=("battery_id","size"),
        drop_pm=("drop_per_month","mean"), retired=("retired","mean")).sort_values("drop_pm", ascending=False)
print(lot.head(10))                                       # outlier lots
print(pd.crosstab(bat.supplier, bat.retirement_reason.fillna("active"), normalize="index").round(3))

                    packs  start_soh    now_soh   drop_pm   retired  \
supplier pack_type                                                    
Amptek   2W_2.1kWh   1098  99.192441  79.956648  4.016986  0.000000   
         3W_4.8kWh    209  99.204785  86.513876  2.780164  0.000000   
Cellora  2W_2.1kWh   3035  99.181483  79.970675  2.315975  0.000000   
         3W_4.8kWh    520  99.197115  86.522500  1.727174  0.000000   
Kyron    2W_2.1kWh   1385  99.197473  60.532274  4.951843  1.000000   
         3W_4.8kWh    253  99.247036  81.348221  3.807270  0.209486   

                            cost  
supplier pack_type                
Amptek   2W_2.1kWh  31995.052823  
         3W_4.8kWh  67822.258373  
Cellora  2W_2.1kWh  31983.387809  
         3W_4.8kWh  67957.390385  
Kyron    2W_2.1kWh  31981.684477  
         3W_4.8kWh  67920.948617  
                            packs    drop_pm   retired
supplier manufacturing_lot                            
Amptek   AM-2506               47  18.100

In [16]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Fit the Ordinary Least Squares model
model = smf.ols("drop_per_month ~ C(supplier) + C(pack_type) + age_months", data=bat).fit()

# Print the full summary instead of indexing into tables[1]
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         drop_per_month   R-squared:                       0.504
Model:                            OLS   Adj. R-squared:                  0.504
Method:                 Least Squares   F-statistic:                     1652.
Date:                Sun, 15 Nov 2026   Prob (F-statistic):               0.00
Time:                        13:11:36   Log-Likelihood:                -14040.
No. Observations:                6500   AIC:                         2.809e+04
Df Residuals:                    6495   BIC:                         2.812e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [17]:
bat.to_csv("processed/batteries_clean.csv", index=False)

In [19]:
fp = pd.read_csv(f"fleet_partners.csv", parse_dates=["contract_start_date","amendment_date"])
fp["discount_change_pp"] = (fp.discount_pct_after_amendment - fp.discount_pct)   # may be NaN
fp["n_cities"] = fp.cities_active.str.count(r"\|") + 1
print(fp.sort_values("discount_pct", ascending=False))
print(fp.groupby("contract_type")[["discount_pct","payment_terms_days"]].mean())
print(fp.groupby("peak_surcharge_billable").size())

   partner_id partner_name partner_segment vehicle_class  contract_type  \
1       FP-02   ParcelNest  ecom_logistics            2W   pay_per_swap   
2       FP-03      ZipDrop  quick_commerce            2W  volume_tiered   
9       FP-10    MetroMove  ecom_logistics         mixed  volume_tiered   
4       FP-05   Swiggle Go   food_delivery            2W  volume_tiered   
10      FP-11  LastMileHub  ecom_logistics            2W   pay_per_swap   
0       FP-01     FeastFly   food_delivery            2W  volume_tiered   
6       FP-07    QuickCart  quick_commerce            2W   pay_per_swap   
3       FP-04     CargoTuk        cargo_3w            3W   pay_per_swap   
7       FP-08  DabbaXpress   food_delivery            2W   pay_per_swap   
8       FP-09     HaulKing        cargo_3w            3W   pay_per_swap   
5       FP-06    RideMitra       bike_taxi            2W   pay_per_swap   
11      FP-12  UrbanErrand       bike_taxi            2W   pay_per_swap   

   contract_start_date  

In [20]:
fp.to_csv("processed/partners_clean.csv", index=False)

In [22]:
riders = pd.read_csv(f"riders.csv", parse_dates=["signup_date"])
print(riders.home_city.value_counts())          # inspect all variants first

def norm_city(s):
    s = str(s).lower().strip()
    for k, v in [("bang","Bengaluru"),("blr","Bengaluru"),("bengal","Bengaluru"),
                 ("delhi","Delhi NCR"),("ncr","Delhi NCR"),("gurug","Delhi NCR"),("gurgaon","Delhi NCR"),("noida","Delhi NCR"),
                 ("mum","Mumbai"),("bom","Mumbai"),("hyd","Hyderabad"),("pun","Pune"),("jai","Jaipur")]:
        if k in s: return v
    return "UNMAPPED:" + s
riders["city"] = riders.home_city.map(norm_city)
print(riders.city.value_counts())               # must be exactly six cities, no UNMAPPED

riders["fleet"] = riders.partner_id.notna()
riders["signup_month"] = riders.signup_date.dt.to_period("M").astype(str)

print(pd.crosstab(riders.signup_month, riders.signup_channel))            # acquisition mix over time
print(pd.crosstab(riders.plan_type, riders.fleet))
print(pd.crosstab(riders.vehicle_class, riders.fleet, normalize="index").round(3))
print(riders.groupby("partner_id").size().sort_values(ascending=False))   # partner exposure
print(riders.kyc_verified.mean(), riders.declared_shift.isna().mean())

home_city
Bengaluru     4175
Delhi NCR     3783
Hyderabad     3396
Pune          2843
Mumbai        2772
Jaipur        2446
Bombay          53
BLR             53
bengaluru       50
MUM             49
pune            43
PUN             43
New Delhi       42
JAI             41
Gurgaon         36
Bangalore       32
HYD             31
Hyd             31
jaipur          29
hyderabad       27
Delhi           25
Name: count, dtype: int64
city
Bengaluru    4310
Delhi NCR    3886
Hyderabad    3485
Pune         2929
Mumbai       2874
Jaipur       2516
Name: count, dtype: int64
signup_channel  app_store  field_agent  partner_onboarding  referral
signup_month                                                        
2023-03                71           72                 154        26
2023-04                87           83                 200        36
2023-05                72           79                 180        36
2023-06                92           83                 216        36
2023-07     

In [23]:
riders.to_csv("processed/riders_clean.csv", index=False)

In [25]:
tix = pd.read_csv(f"support_tickets.csv", parse_dates=["created_ts"])
tix["month"] = tix.created_ts.dt.to_period("M").astype(str)
print(tix.category.value_counts(normalize=True).round(3))
print(tix.resolution_status.value_counts(normalize=True).round(3))
print(pd.crosstab(tix.month, tix.category, normalize="index").round(3))     # category mix over time
print(tix.groupby("category").resolution_hours.median())

# CSAT is missing-not-at-random: show the bias
tix["has_csat"] = tix.csat_score.notna()
print(tix.groupby(pd.cut(tix.resolution_hours,[0,4,12,24,48,1e9])).has_csat.mean())

# hidden battery issues: generic categories that nonetheless reference a battery
tix["has_battery"] = tix.battery_id.notna()
print(pd.crosstab(tix.category, tix.has_battery, normalize="index").round(3))
print(tix.groupby("rider_id").size().describe())

category
no_battery_available    0.265
other                   0.216
low_range               0.151
app_issue               0.132
long_queue              0.130
billing_dispute         0.107
Name: proportion, dtype: float64
resolution_status
resolved      0.861
unresolved    0.118
duplicate     0.021
Name: proportion, dtype: float64
category  app_issue  billing_dispute  long_queue  low_range  \
month                                                         
2023-03       0.000            0.000       0.333      0.333   
2023-04       0.000            0.000       0.000      0.000   
2023-05       0.000            0.000       0.000      0.000   
2023-07       0.000            0.000       0.000      0.000   
2023-08       0.000            0.500       0.000      0.000   
2023-09       0.000            0.000       1.000      0.000   
2023-10       0.000            0.000       0.000      0.000   
2023-11       0.000            1.000       0.000      0.000   
2023-12       0.000            1.000 

/tmp/ipykernel_550/3560817975.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tix.groupby(pd.cut(tix.resolution_hours,[0,4,12,24,48,1e9])).has_csat.mean())


In [26]:
tix.to_csv("processed/tickets_clean.csv", index=False)

In [27]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Loading clean datasets from the processed folder
bat = pd.read_csv("processed/batteries_clean.csv")
riders = pd.read_csv("processed/riders_clean.csv")
stn = pd.read_csv("processed/stations_clean.csv")
fp = pd.read_csv("processed/partners_clean.csv")
tix = pd.read_csv("processed/tickets_clean.csv")


# TEST 1: PEARSON CORRELATION (Numerical Data)

print(" 1. PEARSON CORRELATION TESTS")
# Let's check numerical correlations in batteries.csv (e.g., age vs health drop)
bat_numeric = bat[["initial_soh_pct", "current_soh_pct", "rated_capacity_kwh", "purchase_cost_inr", "age_months", "soh_drop"]].dropna()
corr_matrix = bat_numeric.corr(method="pearson")
print("Battery Correlation Matrix:")
print(corr_matrix.round(3))
print("\n")

# TEST 2: CHI-SQUARE TESTS (Categorical Data)
print(" 2. CHI-SQUARE DEPENDENCY TESTS")

# Test A: Is a rider's plan_type dependent on whether they are in a fleet? (from riders.csv)
contingency_table_1 = pd.crosstab(riders["plan_type"], riders["fleet"])
chi2_1, p_val_1, dof_1, expected_1 = chi2_contingency(contingency_table_1)
print(f"Test A (Plan Type vs Fleet Status) -> Chi2: {chi2_1:.2f}, p-value: {p_val_1:.4f}")
if p_val_1 < 0.05:
    print("-> Result: Statistically significant dependency! Plan type choices are strongly related to fleet status.")
else:
    print("-> Result: Independent.")

print("-" * 40)

# Test B: Do charger generations map to specific connectivity tiers? (from stations.csv)[cite: 10]
contingency_table_2 = pd.crosstab(stn["charger_generation"], stn["connectivity_tier"])
chi2_2, p_val_2, dof_2, expected_2 = chi2_contingency(contingency_table_2)
print(f"Test B (Charger Generation vs Connectivity Tier) -> Chi2: {chi2_2:.2f}, p-value: {p_val_2:.4f}")
if p_val_2 < 0.05:
    print("-> Result: Statistically significant dependency! (Confirms the connectivity confounder trap mentioned in Table 1).")
else:
    print("-> Result: Independent.")

 1. PEARSON CORRELATION TESTS
Battery Correlation Matrix:
                    initial_soh_pct  current_soh_pct  rated_capacity_kwh  \
initial_soh_pct               1.000            0.042               0.022   
current_soh_pct               0.042            1.000               0.410   
rated_capacity_kwh            0.022            0.410               1.000   
purchase_cost_inr             0.024            0.407               0.994   
age_months                   -0.008            0.240              -0.048   
soh_drop                      0.003           -0.999              -0.409   

                    purchase_cost_inr  age_months  soh_drop  
initial_soh_pct                 0.024      -0.008     0.003  
current_soh_pct                 0.407       0.240    -0.999  
rated_capacity_kwh              0.994      -0.048    -0.409  
purchase_cost_inr               1.000      -0.048    -0.406  
age_months                     -0.048       1.000    -0.240  
soh_drop                       -0.406